In [ ]:
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
from datasets import load_dataset, load_metric
from transformers import MarianMTModel, MarianTokenizer
from comet import download_model, load_from_checkpoint
import torch

# -----------------------
# Setup
# -----------------------
torch.backends.mps.is_available = lambda: False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name).to(device)
model.eval()

print(f"Loaded translation model: {model_name}")

# -----------------------
# Load IWSLT dataset (English–French)
# -----------------------
dataset = load_dataset("iwslt2017", "iwslt2017-en-fr")
test_data = dataset["test"]

df = pd.DataFrame({
    "src": [ex["translation"]["en"] for ex in test_data],
    "tgt": [ex["translation"]["fr"] for ex in test_data]
})

# Limit to first 300 samples for speed
df = df.head(300).reset_index(drop=True)
print(f"Loaded {len(df):,} sentence pairs from IWSLT17 (English–French).")

# -----------------------
# Sentence-level translation (no chunking)
# -----------------------
preds, latencies = [], []

print("\nTranslating full sentences (baseline)...")

for src in tqdm(df["src"], desc="Full-sentence translation"):
    start = time.time()
    inputs = tokenizer(src, return_tensors="pt", truncation=True).to(device)
    outputs = model.generate(**inputs, max_length=256)
    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    end = time.time()

    preds.append(pred)
    latencies.append(end - start)

df["pred"] = preds
df["latency_sec"] = latencies

# Save results
output_path = "IWSLT_full_sentence_translated.csv"
df.to_csv(output_path, index=False, encoding="utf-8")
print(f"\nSaved translated results with latency to {output_path}")

# -----------------------
# Evaluation
# -----------------------
metric_bleu = load_metric("sacrebleu")
metric_chrf = load_metric("chrf")

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

src_texts = df["src"].tolist()
tgt_texts = df["tgt"].tolist()
preds = df["pred"].tolist()

# BLEU / chrF
bleu = metric_bleu.compute(predictions=preds, references=[[t] for t in tgt_texts])
chrf = metric_chrf.compute(predictions=preds, references=[[t] for t in tgt_texts])

# COMET
data = [{"src": s, "mt": p, "ref": t} for s, p, t in zip(src_texts, preds, tgt_texts)]
comet_score = comet_model.predict(data, batch_size=4, gpus=0, num_workers=0)
comet_mean = np.mean(comet_score["system_score"])

# Latency
avg_latency = df["latency_sec"].mean()

# -----------------------
# Print results
# -----------------------
print("\n===== Full-Sentence Baseline Evaluation =====")
print(f"BLEU: {bleu['score']:.2f}")
print(f"chrF: {chrf['score']:.2f}")
print(f"COMET: {comet_mean:.4f}")
print(f"Latency (sec/sentence): {avg_latency:.3f}")
print("==============================================")

# Save summary
summary = pd.DataFrame([{
    "Setting": "Full-sentence (baseline)",
    "BLEU": bleu["score"],
    "chrF": chrf["score"],
    "COMET": comet_mean,
    "Latency (sec/sentence)": avg_latency
}])

summary.to_csv("IWSLT_full_sentence_evaluation_summary.csv", index=False)
print(f"\nSaved evaluation summary to IWSLT_full_sentence_evaluation_summary.csv")
